In [6]:
import json
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
import torch
from unsloth import FastLanguageModel
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 配置路径
TEST_DATA_PATH = "./yue/yue_test.jsonl"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f]

total_score = 0
smooth = SmoothingFunction().method1

print(f"📊 正在对 2.1 万条测试集进行裸考评估...")
# 先跑 500 条快速看分，如果你想跑全量，把 [:500] 去掉
for item in tqdm(test_cases[:500]):
    prompt = f"<|im_start|>system\n你是一个地道的粤语翻译助手。<|im_end|>\n<|im_start|>user\n{item['input']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
    pred = tokenizer.batch_decode(outputs)[0].split("assistant\n")[-1].replace("<|im_end|>", "").strip()
    
    ref = [list(item['output'])]
    hyp = list(pred)
    total_score += sentence_bleu(ref, hyp, smoothing_function=smooth)

print(f"\n✨ 基准测试完成！当前平均 BLEU: {total_score/500:.4f}")

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:40<00:00,  8.38it/s]


unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
📊 正在对 2.1 万条测试集进行裸考评估...


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAG


✨ 基准测试完成！当前平均 BLEU: 0.1759
